In [0]:
%load_ext autoreload
%autoreload 2

import os
import sys

sys.path.append(os.path.abspath("../src"))
from business_vault import PITS, statements

In [0]:
dbutils.widgets.text("catalog", "reinsurance")
catalog = dbutils.widgets.get("catalog")

for sql in statements(f"{catalog}.raw_vault", f"{catalog}.business_vault"):
    spark.sql(sql)

display(spark.sql(" UNION ALL ".join(
    f"SELECT '{t}' AS pit, count(*) AS rows, min(snapshot_date) AS first_month, max(snapshot_date) AS last_month "
    f"FROM {catalog}.business_vault.{t}" for t in PITS
)))

In [0]:
display(spark.sql(f"""
SELECT coalesce(treaty_type, 'NO TREATY') AS treaty_type, count(*) AS claims,
       sum(gross_incurred) AS gross, sum(ceded_incurred) AS ceded,
       round(sum(ceded_incurred) / sum(gross_incurred), 3) AS ceded_ratio
FROM {catalog}.business_vault.claim_ceded
WHERE snapshot_date = (SELECT max(snapshot_date) FROM {catalog}.business_vault.claim_ceded)
GROUP BY 1
ORDER BY 1"""))

In [0]:
display(spark.sql(f"""
SELECT count(*) AS reviewed,
       sum(CASE WHEN p.predicted_cause = r.cause THEN 1 ELSE 0 END) AS correct,
       round(avg(CASE WHEN p.predicted_cause = r.cause THEN 1.0 ELSE 0.0 END), 3) AS accuracy
FROM {catalog}.raw_vault.sat_claim_cause_review r
JOIN {catalog}.business_vault.claim_cause p ON p.hk_claim = r.hk_claim"""))

In [0]:
display(spark.sql(f"""
SELECT r.cause AS reviewed_cause, p.predicted_cause, p.note
FROM {catalog}.raw_vault.sat_claim_cause_review r
JOIN {catalog}.business_vault.claim_cause p ON p.hk_claim = r.hk_claim
WHERE p.predicted_cause <> r.cause
ORDER BY r.cause"""))